# DCGAN Training on FFHQ 128×128

**Архитектура:**  
- **Generator:** ConvTranspose + Self-Attention на 64×64  
- **Discriminator:** Conv + Spectral Normalization  

**Loss:** Binary Cross Entropy с Label Smoothing  
**Датасет:** FFHQ thumbnails 128×128

> ⚠️ **GAN нестабильны.** Следи за D(x) ≈ 0.5–0.7 и D(G(z)) ≈ 0.3–0.5.  
> Если D(G(z)) → 1.0 — возможен mode collapse.

---

## ⚙️ Конфигурация

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

DATA_ROOT    = '../data'
OUTPUT_DIR   = '../checkpoints/gan'

LATENT_DIM   = 100
NGF          = 64    # Generator фильтры
NDF          = 64    # Discriminator фильтры

EPOCHS       = 100
BATCH_SIZE   = 32
LR_G         = 2e-4
LR_D         = 2e-4
BETA1        = 0.5   # Adam β₁ (из статьи DCGAN)
BETA2        = 0.999
LABEL_SMOOTH = 0.1   # реальные → [0.9, 1.0]
N_CRITIC     = 1     # шагов D на 1 шаг G

SAVE_EVERY   = 10
SAMPLE_EVERY = 5
LOG_EVERY    = 20
N_SAMPLES    = 64

DRY_RUN      = False
RESUME       = None

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 📂 Загрузка данных

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

if DRY_RUN:
    class _Dummy:
        def __init__(self): self.n = 10
        def __len__(self): return self.n
        def __iter__(self):
            for _ in range(self.n): yield torch.randn(BATCH_SIZE, 3, 128, 128)
    train_loader = _Dummy()
    print('⚠️  DRY RUN')
else:
    from data.dataset import get_dataloaders
    train_loader, _ = get_dataloaders(
        data_root=DATA_ROOT, batch_size=BATCH_SIZE, num_workers=4,
    )

print(f'Train батчей: {len(train_loader)}')

plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                     'text.color': '#c9d1d9', 'axes.titlecolor': '#c9d1d9'})

sample_batch = next(iter(train_loader))[:16]
grid = make_grid((sample_batch + 1) / 2, nrow=8, padding=2, pad_value=0.1)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title('FFHQ — Real Images (preview)', fontsize=11)
plt.tight_layout()
plt.show()

## 🏗️ Модели

In [ ]:
from GAN.gan import Generator, Discriminator, weights_init, smooth_labels
from utils.utils import print_model_info

G = Generator(latent_dim=LATENT_DIM, ngf=NGF).to(DEVICE)
D = Discriminator(nc=3, ndf=NDF).to(DEVICE)
G.apply(weights_init)
D.apply(weights_init)

print_model_info(G, f'Generator  (latent={LATENT_DIM}, ngf={NGF})')
print_model_info(D, f'Discriminator (Spectral Norm, ndf={NDF})')

# Тест
with torch.no_grad():
    z_test  = torch.randn(2, LATENT_DIM, 1, 1, device=DEVICE)
    fake    = G(z_test)
    score   = D(fake)
print(f'✓ G output: {fake.shape}  |  D output: {score.shape}')

# Фиксированный шум для мониторинга прогресса
FIXED_NOISE = torch.randn(N_SAMPLES, LATENT_DIM, 1, 1, device=DEVICE)

## 🚀 Обучение

In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from collections import defaultdict

from utils.utils import (
    TrainingLogger, Visualizer,
    save_checkpoint, load_checkpoint, save_sample_grid,
)

torch.manual_seed(42)

criterion = nn.BCELoss()
opt_g = optim.Adam(G.parameters(), lr=LR_G, betas=(BETA1, BETA2))
opt_d = optim.Adam(D.parameters(), lr=LR_D, betas=(BETA1, BETA2))

logger = TrainingLogger(OUTPUT_DIR, model_name='GAN')
vis    = Visualizer(OUTPUT_DIR, model_name='DCGAN', inline=True)

start_epoch = 1
if RESUME and os.path.exists(RESUME):
    ckpt = load_checkpoint(RESUME, device=str(DEVICE))
    G.load_state_dict(ckpt['G']); D.load_state_dict(ckpt['D'])
    opt_g.load_state_dict(ckpt['opt_g'])
    opt_d.load_state_dict(ckpt['opt_d'])
    start_epoch = ckpt.get('epoch', 0) + 1

_epochs = 2 if DRY_RUN else EPOCHS

for epoch in range(start_epoch, _epochs + 1):
    G.train(); D.train()
    d_acc = g_acc = dx_acc = dgz_acc = 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:3d}/{_epochs}', leave=False)
    for step, real in enumerate(pbar):
        real = real.to(DEVICE, non_blocking=True)
        B    = real.size(0)
        dev  = str(DEVICE)

        # ── D шаг ──────────────────────────────────────────────────────
        for _ in range(N_CRITIC):
            D.zero_grad()
            real_lbl = smooth_labels(B, real=True,  smooth=LABEL_SMOOTH, device=dev)
            d_real   = D(real)
            loss_dr  = criterion(d_real, real_lbl)

            z    = torch.randn(B, LATENT_DIM, 1, 1, device=DEVICE)
            fake = G(z).detach()
            fake_lbl = smooth_labels(B, real=False, smooth=LABEL_SMOOTH, device=dev)
            d_fake   = D(fake)
            loss_df  = criterion(d_fake, fake_lbl)

            loss_d = (loss_dr + loss_df) * 0.5
            loss_d.backward()
            opt_d.step()

        # ── G шаг ──────────────────────────────────────────────────────
        G.zero_grad()
        z    = torch.randn(B, LATENT_DIM, 1, 1, device=DEVICE)
        fake = G(z)
        g_lbl = torch.ones(B, device=DEVICE)
        d_g2  = D(fake)
        loss_g = criterion(d_g2, g_lbl)
        loss_g.backward()
        opt_g.step()

        d_acc  += loss_d.item(); g_acc  += loss_g.item()
        dx_acc += d_real.mean().item()
        dgz_acc+= d_g2.mean().item()
        pbar.set_postfix(D=f'{loss_d.item():.3f}', G=f'{loss_g.item():.3f}',
                         Dx=f'{d_real.mean().item():.3f}',
                         DGz=f'{d_g2.mean().item():.3f}')

    n = len(train_loader)
    metrics = {'d_loss': d_acc/n, 'g_loss': g_acc/n,
               'D_x': dx_acc/n, 'D_G_z': dgz_acc/n}
    logger.log_epoch(epoch=epoch, **metrics)

    # Предупреждение
    warn = ''
    if metrics['D_x'] < 0.3: warn += ' ⚠️D_слабый'
    if metrics['D_G_z'] > 0.8: warn += ' ⚠️mode_collapse?'

    print(f'Epoch {epoch:3d}/{_epochs}  |  '
          f'D={metrics["d_loss"]:.4f}  G={metrics["g_loss"]:.4f}  '
          f'D(x)={metrics["D_x"]:.3f}  D(G(z))={metrics["D_G_z"]:.3f}{warn}')

    # ── Образцы ────────────────────────────────────────────────────────
    if epoch % SAMPLE_EVERY == 0 or epoch == _epochs:
        G.eval()
        with torch.no_grad():
            fake_imgs = G(FIXED_NOISE)
        save_sample_grid(fake_imgs,
            path=f'{OUTPUT_DIR}/samples/gen_ep{epoch:03d}.png',
            nrow=8, title=f'DCGAN — Epoch {epoch}')
        G.train()

    # ── Чекпоинт ───────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0 or epoch == _epochs:
        save_checkpoint(
            {'epoch': epoch, 'G': G.state_dict(), 'D': D.state_dict(),
             'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict()},
            OUTPUT_DIR, filename=f'checkpoint_ep{epoch:03d}.pt',
        )

    # ── Live plot ──────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0 or epoch == _epochs:
        vis.plot_curves(logger.epoch_history, epoch=epoch, save=True, show=True)

logger.close()
print('\n✅ Обучение завершено!')

## 📊 Финальные результаты

In [ ]:
G.eval()
with torch.no_grad():
    final_samples = G(FIXED_NOISE)

poster_path = vis.plot_final_summary(
    logger.epoch_history, samples=final_samples,
)

from IPython.display import Image as IPyImage
IPyImage(poster_path)

In [ ]:
# ── Проверка разнообразия генерации ───────────────────────────────────
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                     'text.color': '#c9d1d9'})

G.eval()
with torch.no_grad():
    z_diverse = torch.randn(32, LATENT_DIM, 1, 1, device=DEVICE)
    imgs = G(z_diverse)

grid = make_grid((imgs.cpu() + 1) / 2, nrow=8, padding=2, pad_value=0.1)
fig, ax = plt.subplots(figsize=(18, 6))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title('DCGAN Generated — Diversity Check (32 samples)', color='#58a6ff', fontsize=12)
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/diversity.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()